In [ ]:
from datasets import Dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re


In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer,DataCollatorForSeq2Seq, Seq2SeqTrainingArguments
import torch
from evaluate import load

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())       # True if GPU is usable
print("CUDA version:", torch.version.cuda)                # CUDA version PyTorch was built with
print("Number of GPUs:", torch.cuda.device_count())       # Number of GPUs detected
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## Importing and cleaning data set

In [ ]:
documention_path_0 = "document_train_0.parquet" #file path
document_0 = pd.read_parquet(documention_path_0)

documention_path_1 = "document_train_1.parquet"
document_1 = pd.read_parquet(documention_path_1)

documention_path_2 = "document_train_2.parquet"
document_2 = pd.read_parquet(documention_path_2)

documention_path_3 = "document_train_3.parquet"
document_3 = pd.read_parquet(documention_path_3)

In [ ]:
document = pd.concat([document_0, document_1, document_2, document_3])

In [ ]:
document

In [ ]:
def clean_document(document):
    document['abstract'] = document['abstract'].str.replace(' \n', '', regex=False) #remove \n
    document['abstract'] = document['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['abstract'] = document['abstract'].str.replace('  ', ' ', regex=False) #replace double space with single space

    document['article'] = document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['article'] = document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

    article_summary = document['article'].str.len().describe()
    abstract_summary = document['abstract'].str.len().describe()

    document = document[document['article'].str.len() >= article_summary['25%']] # has to be at least 25% article length
    document = document[document['article'].str.len() <= article_summary['75%']] # has to be at most 75% article length

    document = document[document['abstract'].str.len() >= abstract_summary['25%']] 
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document[document['article'].str.len() >= article_summary['25%']] 
    document = document[document['article'].str.len() <= article_summary['75%']]  

    document = document[document['abstract'].str.len() >= abstract_summary['25%']]
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document.drop_duplicates()
    document = document.reset_index(drop=True)
    
    return document

In [ ]:
document = clean_document(document)

## Validation and Test cleaning

In [ ]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True, torch_dtype = torch.bfloat16)

In [ ]:
validation_path = "document_validation.parquet" #file path
validation = pd.read_parquet(validation_path)

test_path = "document_test.parquet"
test = pd.read_parquet(test_path)

In [ ]:
validation = clean_document(validation)

In [ ]:
test = clean_document(test)

## Training

In [ ]:
train_dataset = Dataset.from_pandas(document)
validation_dataset   = Dataset.from_pandas(validation)
test_dataset  = Dataset.from_pandas(test)

In [ ]:
label = tokenizer(
    list(train_dataset["abstract"]),
    truncation=True,
    max_length=1024,
    padding=False
)

# Get lengths
label_lengths = [len(ids) for ids in label["input_ids"]]

In [ ]:
abstract_tokens_summary = pd.DataFrame(label_lengths).describe()[0]
abstract_tokens_summary

In [ ]:
token_len_IQR =  abstract_tokens_summary['75%'] - abstract_tokens_summary['25%']
1.5 * token_len_IQR

In [ ]:
abstract_tokens_summary['25%'] - (1.5 * token_len_IQR)

In [ ]:
abstract_tokens_summary['75%'] + (1.5 * token_len_IQR)

#pad to 320 should be a clean number

In [ ]:
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746").cuda()
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

In [ ]:
def tokenize_document(dataset):
    model_input = tokenizer(dataset["article"], max_length = 1024, truncation = True)
    labels = tokenizer(dataset["abstract"], max_length = 320, truncation = True)

    model_input["labels"] = labels['input_ids']
    return model_input

In [ ]:
tokenized_dataset = train_dataset.map(tokenize_document, batched=True)
validation_tokenized = validation_dataset.map(tokenize_document, batched=True, remove_columns=validation_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_document, batched=True, remove_columns=test_dataset.column_names)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest"  # pads to the longest sequence in each batch
)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",      # folder to save checkpoints
    num_train_epochs=3,                  # number of passes through the dataset
    per_device_train_batch_size=4,       # safe for 12GB VRAM
    per_device_eval_batch_size=4,        # same as train batch size
    learning_rate=2e-5,                  # learning rate
    weight_decay=0.01,                   # regularization
    save_total_limit=3,                   # keep last 3 checkpoints
    eval_strategy="epoch",          # evaluate once per epoch
    save_strategy="epoch",                # save checkpoint once per epoch
    logging_strategy="steps",             # log every N steps
    logging_steps=100,                    # adjust if needed
    bf16=True,                            # mixed precision for VRAM efficiency
    predict_with_generate=True            # necessary for seq2seq evaluation
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint="./bart_finetuned/checkpoint-7746") #pick up from Epoch = 2

### Epoch	Training Loss	Validation Loss
### 1	2.795941	2.686332
### 2	2.850331	2.683255
### 3	2.777663	2.682559

# Summary Generation comparison

In [ ]:
test_section = pd.read_parquet('section_test_0.parquet')

test_section['article'] = test_section['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
test_section['article'] = test_section['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

test_document = pd.read_parquet('document_test.parquet')

test_document['article'] = test_document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
test_document['article'] = test_document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-11619", attn_implementation="eager").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-11619")

In [ ]:
test_ind = 9

In [ ]:
inputs = tokenizer(
    test_document['article'][test_ind],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=1024   # BART max input length
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=320,
        min_length=30,
        num_beams=1,
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

In [ ]:
test_document['abstract'][test_ind]

In [ ]:
test_document['article'][test_ind]

# Cross Attention

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #prioritze cuda
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-11619", attn_implementation="eager").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-11619")

In [ ]:
def get_token_score_pairs(article, model, tokenizer):

    inputs = tokenizer(
        article,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024
    ).to('cuda')

    outputs = model(
        **inputs,
        output_attentions=True,
        return_dict=True
    )

    cross_attentions = outputs.cross_attentions
    last_layer = cross_attentions[-1]
    avg_heads = last_layer.mean(dim=1)
    token_importance = avg_heads.mean(dim=1)

    input_ids = inputs["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    token_scores = [(token, float(score.detach())) for token, score in zip(tokens, token_importance[0])]

    return token_scores

In [ ]:
def rank_important_lines(token_scores, average_score = False):
    """"
    This function ranks lines based on their importance divided by the amount of tokens
    This only works if we use a sectioned article with \n as we will use that to detect new lines

    Parameters
    ----------
    token_scores: list of tuples (token, score)
        token_scores is a list of tuples where each tuple contains a token and its corresponding importance score
    
    average_score: bool (default False)
        if True then it will rank based on the average score of all tokens in a line
        if False then it will rank based on the sum of all tokens in a line

    Returns
    -------
    list of tuples (line, total_score, avg_score, line position)
        this function returns a list of tuples for token line, overall token score, average score across tokens, and where the line is in the article
    """
    lines = []
    current_line = []
    current_score = 0
    line_count = 0

    newline_count = sum(1 for token, _ in token_scores if token == "Ċ")
    total_lines = newline_count + 1

    for token, score in token_scores:
        if token != "Ċ": #this token symbolizes new line so if it isn't a new line keeping adding tokens
            current_line.append(token)
            current_score += score
        else:
            if current_line != []: #check if current line is empty or not
                avg_score = current_score / len(current_line) #calculate average score for the line
                lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score, avg_score, line_count / total_lines)) 
            current_line = []
            current_score = 0
            line_count += 1

    if current_line != []:
        avg_score = current_score / len(current_line) #calculate average score for the line
        lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score, avg_score, line_count / total_lines))

    lines[0]  = (lines[0][0].replace("<s>", "").strip(),  lines[0][1], lines[0][2], lines[0][3]) #take out starting token
    lines[-1] = (lines[-1][0].replace("</s>", "").strip(), lines[-1][1], lines[-1][2], lines[-1][3]) #take out ending token

    if average_score:
        ranked_lines = sorted(lines, key=lambda x: x[2], reverse=True) #ranks the lines based on the average score in desc order
    else:
        ranked_lines = sorted(lines, key=lambda x: x[1], reverse=True) #ranks the lines based on the total score in desc order

    return ranked_lines

In [ ]:
pairs = get_token_score_pairs(test_section['article'][2], model, tokenizer)
rank_important_lines(pairs, average_score= True)

In [ ]:
print(summary)

In [ ]:
test_document['abstract'][test_ind]

In [ ]:
test_document['article'][test_ind]

In [ ]:
most_important_line_aggregate = []

for article in test_section['article']:
    pairs = get_token_score_pairs(article, model, tokenizer)
    lines = rank_important_lines(pairs, average_score=False)
    most_important_line_aggregate.append(lines[0][3])

In [ ]:
most_important_line_aggregate

In [ ]:
plt.hist(most_important_line_aggregate, bins=10, edgecolor='black') 

plt.xlabel('Proportion')
plt.ylabel('Frequency')
plt.title('Most important line')

plt.show()

In [ ]:
most_important_line_average = []

for article in test_section['article']:
    pairs = get_token_score_pairs(article, model, tokenizer)
    lines = rank_important_lines(pairs, average_score=True)
    most_important_line_average.append(lines[0][3])

In [ ]:
most_important_line_average

In [ ]:
plt.hist(most_important_line_average, bins=10, edgecolor='black') 

plt.xlabel('Proportion')
plt.ylabel('Frequency')
plt.title('Most important line on average')

plt.show()

## ROUGE comparison

1. Have the pre trained model and fine tuned model and both should use the same parameters
2. input cleaned or entire dataset into both params
3. calculate rouge 1, rouge 2, rouge L, and rouge L sum for each model
4. take average scores across all generations to get averaged Rouge 1, averaged Rouge ..., and compare

In [ ]:
test['article'][0]

In [ ]:
test['abstract'][0]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-11619").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-11619")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = BartTokenizer.from_pretrained(model_name)

articles = test['article'].to_list()
abstracts = test['abstract'].to_list()

rouge = load("rouge")

batch_size = 2
summaries = []

for i in range(0, len(articles), batch_size):
    batch_articles = articles[i:i+batch_size]

    inputs = tokenizer(
        batch_articles,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024  
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=320,
            min_length=30,
            num_beams=1
        )

    batch_summaries = [tokenizer.decode(s, skip_special_tokens=True) for s in summary_ids]

    summaries.extend(batch_summaries)

results = rouge.compute(predictions=summaries, references=abstracts)
print(results)

## For Epoch = 3 on testing data

{'rouge1': np.float64(0.4255260716239057),
 'rouge2': np.float64(0.15537410540264113),
 'rougeL': np.float64(0.24006861503538948),
 'rougeLsum': np.float64(0.24007864179224242)}

## For base Bart model

{'rouge1': np.float64(0.31840859170721303), 'rouge2': np.float64(0.09706548835740322), 'rougeL': np.float64(0.18876960454777048), 'rougeLsum': np.float64(0.18879169146238267)}


In [ ]:
results

In [ ]:
from evaluate import load

rouge = load('rouge')

predictions = [summary]
references = [test['abstract'][0]]

results = rouge.compute(predictions=predictions, references=references)

print(results)

In [ ]:
model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = BartTokenizer.from_pretrained(model_name)

In [ ]:
inputs = tokenizer(
    test['article'][0],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=1024   # BART max input length
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=320,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

In [ ]:
from evaluate import load

rouge = load('rouge')

predictions = [summary]
references = [test['abstract'][0]]

results = rouge.compute(predictions=predictions, references=references)

print(results)

## BERTscore

same idea as rouge, but for bert score comparison

In [ ]:
bertscore = load("bertscore")

In [ ]:
summary

In [ ]:
test_document['abstract'][test_ind]

In [ ]:
from bert_score import score

candidates = [summary]
references = [test_document['abstract'][test_ind]]

P, R, F1 = score(candidates, references, lang="en", model_type = "roberta-large", device = "cuda", batch_size = 12, verbose= False,)

print(P)
print(R)
print(F1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = BartTokenizer.from_pretrained(model_name)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-11619").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-11619")

articles = test['article'].to_list()
abstracts = test['abstract'].to_list()

batch_size = 2
summaries = []

for i in range(0, len(articles), batch_size):

    batch_articles = articles[i:i+batch_size]

    inputs = tokenizer(
        batch_articles,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=320,
            min_length=30,
            num_beams=1
        )

    batch_summaries = [tokenizer.decode(s, skip_special_tokens=True) for s in summary_ids]

    summaries.extend(batch_summaries)

P, R, F1 = score(summaries, abstracts, model_type = "roberta-large", device = "cuda", batch_size = 18, verbose = False)

In [ ]:
print(P.mean())
print(R.mean())
print(F1.mean())

# Base line

BERTScore Precision: 0.858344554901123
BERTScore Recall: 0.8108448386192322
BERTScore F1: 0.833744466304779

# Fine tuned

BERTScore Precision: 0.8467533588409424
BERTScore Recall: 0.8413063287734985
BERTScore F1: 0.8438482880592346